In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 

from statsmodels.graphics.tsaplots import acf, pacf, plot_acf, plot_pacf
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn import metrics

from scikeras.wrappers import KerasRegressor
from keras.models import Sequential
from keras.layers import Dense, Dropout,LSTM
from keras.callbacks import EarlyStopping


import warnings
warnings.filterwarnings("ignore")

In [2]:
features_df = pd.read_csv("../data/Features_dataset.csv")
features_df.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,05/02/2010,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,12/02/2010,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,19/02/2010,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,26/02/2010,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,05/03/2010,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [3]:
stores_df = pd.read_csv("../data/Stores_dataset.csv")
stores_df.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [4]:
sales_df = pd.read_csv("../data/Sales_dataset.csv")
sales_df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,05/02/2010,24924.50,False
1,1,1,12/02/2010,46039.49,True
2,1,1,19/02/2010,41595.55,False
3,1,1,26/02/2010,19403.54,False
4,1,1,05/03/2010,21827.90,False


# Data Pre-Preparation

In [5]:
df = features_df.merge(stores_df, on="Store")
df = sales_df.merge(df, on=["Store", "Date", "IsHoliday"])
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,05/02/2010,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,12/02/2010,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,19/02/2010,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,26/02/2010,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,05/03/2010,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  str    
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
 5   Temperature   421570 non-null  float64
 6   Fuel_Price    421570 non-null  float64
 7   MarkDown1     150681 non-null  float64
 8   MarkDown2     111248 non-null  float64
 9   MarkDown3     137091 non-null  float64
 10  MarkDown4     134967 non-null  float64
 11  MarkDown5     151432 non-null  float64
 12  CPI           421570 non-null  float64
 13  Unemployment  421570 non-null  float64
 14  Type          421570 non-null  str    
 15  Size          421570 non-null  int64  
dtypes: bool(1), float64(10), int64(3), str(2)
memory usage: 48.6 MB


In [7]:
df = df.fillna(0)

In [8]:
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
df["Type"] = df["Type"].astype("category")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         421570 non-null  int64         
 1   Dept          421570 non-null  int64         
 2   Date          421570 non-null  datetime64[us]
 3   Weekly_Sales  421570 non-null  float64       
 4   IsHoliday     421570 non-null  bool          
 5   Temperature   421570 non-null  float64       
 6   Fuel_Price    421570 non-null  float64       
 7   MarkDown1     421570 non-null  float64       
 8   MarkDown2     421570 non-null  float64       
 9   MarkDown3     421570 non-null  float64       
 10  MarkDown4     421570 non-null  float64       
 11  MarkDown5     421570 non-null  float64       
 12  CPI           421570 non-null  float64       
 13  Unemployment  421570 non-null  float64       
 14  Type          421570 non-null  category      
 15  Size          421570 non-nul

In [11]:
df.groupby(["Store", "Dept", "Date"]).sum(numeric_only=True)

Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
Store Dept Date                                                           
1     1    2010-02-05      24924.50          0        42.31       2.572   
           2010-02-12      46039.49          1        38.51       2.548   
           2010-02-19      41595.55          0        39.93       2.514   
           2010-02-26      19403.54          0        46.63       2.561   
           2010-03-05      21827.90          0        46.50       2.625   
...                             ...        ...          ...         ...   
45    98   2012-09-28        508.37          0        64.88       3.997   
           2012-10-05        628.10          0        64.89       3.985   
           2012-10-12       1061.02          0        54.47       4.000   
           2012-10-19        760.01          0        56.47       3.969   
           2012-10-26       1076.80          0        58.85       3.882   

                       MarkDown1  MarkDown2  MarkDown3  MarkDown4  MarkDown5  \
Store Dept Date                                                                
1     1    2010-02-05       0.00       0.00       0.00       0.00       0.00   
           2010-02-12       0.00       0.00       0.00       0.00       0.00   
           2010-02-19       0.00       0.00       0.00       0.00       0.00   
           2010-02-26       0.00       0.00       0.00       0.00       0.00   
           2010-03-05       0.00       0.00       0.00       0.00       0.00   
...                          ...        ...        ...        ...        ...   
45    98   2012-09-28    4556.61      20.64       1.50    1601.01    3288.25   
           2012-10-05    5046.74       0.00      18.82    2253.43    2340.01   
           2012-10-12    1956.28       0.00       7.89     599.32    3990.54   
           2012-10-19    2004.02       0.00       3.18     437.73    1537.49   
           2012-10-26    4018.91      58.08     100.00     211.94     858.33   

                              CPI  Unemployment    Size  
Store Dept Date                                          
1     1    2010-02-05  211.096358         8.106  151315  
           2010-02-12  211.242170         8.106  151315  
           2010-02-19  211.289143         8.106  151315  
           2010-02-26  211.319643         8.106  151315  
           2010-03-05  211.350143         8.106  151315  
...                           ...           ...     ...  
45    98   2012-09-28  192.013558         8.684  118221  
           2012-10-05  192.170412         8.667  118221  
           2012-10-12  192.327265         8.667  118221  
           2012-10-19  192.330854         8.667  118221  
           2012-10-26  192.308899         8.667  118221  

[421570 rows x 12 columns]

In [12]:
df[['Store', 'Dept']].value_counts()

Store  Dept
1      1       143
       2       143
       3       143
       4       143
       5       143
              ... 
38     35        1
       99        1
42     34        1
43     55        1
44     34        1
Name: count, Length: 3331, dtype: int64

In [13]:
st = 24
dt = 50

In [14]:
df_d = df[np.logical_and(df['Store'] == st, df['Dept'] == dt)]